# Stereo Vision Debugging Notebook

In [49]:
import pygame
import ctypes
import serial
import time
from screeninfo import get_monitors
import csv
import pandas as pd

In [50]:
from projection_control import Projector

In [54]:
projector = Projector(display_number=1, square_size=60, delay_ms=500)

projector.run_patterns()

Configuring Projector: 1920x1080
Generating 576 raster patterns...


KeyboardInterrupt: 

In [45]:
import pyrealsense2 as rs

ctx = rs.context()
for dev in ctx.devices:
    print(dev.get_info(rs.camera_info.name), dev.get_info(rs.camera_info.serial_number))

Intel RealSense D455 105322251697
Intel RealSense D455 046322251346


In [40]:
import pyrealsense2 as rs
import numpy as np
import cv2
import os
import time
import argparse

OUTPUT_DIR = "captures"
SERIAL = "105322251697"

def capture_rgb(num_frames: int):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_device(SERIAL)
    config.enable_stream(rs.stream.color, 480, 270, rs.format.bgr8, 15)

    pipeline.start(config)
    time.sleep(2)  # give the camera time to warm up before requesting frames

    try:
        for i in range(num_frames):
            time.sleep(0.1)

            frames = pipeline.wait_for_frames(timeout_ms=1500)
            color_frame = frames.get_color_frame()

            if not color_frame:
                print(f"Frame {i}: no color frame received, skipping.")
                continue

            image = np.asanyarray(color_frame.get_data())
            filename = os.path.join(OUTPUT_DIR, f"{i}.png")
            cv2.imwrite(filename, image)
            print(f"Saved: {filename}")

    finally:
        pipeline.stop()

In [41]:
capture_rgb(10)

Saved: captures\0.png
Saved: captures\1.png
Saved: captures\2.png
Saved: captures\3.png
Saved: captures\4.png
Saved: captures\5.png
Saved: captures\6.png
Saved: captures\7.png
Saved: captures\8.png
Saved: captures\9.png


In [47]:
import pyrealsense2 as rs
import numpy as np
import cv2
import os
import time
import argparse

OUTPUT_DIR = "captures"
CAMERA_1_SERIAL = "105322251697"
CAMERA_2_SERIAL = "046322251346"
CAPTURE_INTERVAL_S = 0.2


def start_pipeline(serial: str):
    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_device(serial)
    config.enable_stream(rs.stream.color, 480, 270, rs.format.bgr8, 5)

    pipeline.start(config)
    return pipeline


def capture_rgb(num_frames: int, projector_update=None, interval_s: float = CAPTURE_INTERVAL_S):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    pipelines = {
        CAMERA_1_SERIAL: start_pipeline(CAMERA_1_SERIAL),
        CAMERA_2_SERIAL: start_pipeline(CAMERA_2_SERIAL),
    }

    time.sleep(2)  # give the cameras time to warm up before requesting frames

    try:
        for i in range(num_frames):
            # if projector_update is not None:
            #     projector_update(i)

            time.sleep(interval_s)

            for serial, pipeline in pipelines.items():
                frames = pipeline.wait_for_frames(timeout_ms=1500)
                color_frame = frames.get_color_frame()

                if not color_frame:
                    print(f"Frame {i} ({serial}): no color frame received, skipping.")
                    continue

                image = np.asanyarray(color_frame.get_data())
                camera_dir = os.path.join(OUTPUT_DIR, serial)
                os.makedirs(camera_dir, exist_ok=True)
                filename = os.path.join(camera_dir, f"{i}.png")
                cv2.imwrite(filename, image)
                print(f"Saved: {filename}")

    finally:
        for pipeline in pipelines.values():
            pipeline.stop()

In [48]:
capture_rgb(10, interval_s=0.2)

Saved: captures\105322251697\0.png
Saved: captures\046322251346\0.png
Saved: captures\105322251697\1.png
Saved: captures\046322251346\1.png
Saved: captures\105322251697\2.png
Saved: captures\046322251346\2.png
Saved: captures\105322251697\3.png
Saved: captures\046322251346\3.png
Saved: captures\105322251697\4.png
Saved: captures\046322251346\4.png
Saved: captures\105322251697\5.png
Saved: captures\046322251346\5.png
Saved: captures\105322251697\6.png
Saved: captures\046322251346\6.png
Saved: captures\105322251697\7.png
Saved: captures\046322251346\7.png
Saved: captures\105322251697\8.png
Saved: captures\046322251346\8.png
Saved: captures\105322251697\9.png
Saved: captures\046322251346\9.png
